In [ ]:
#pip install filterpy


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import os
from sklearn.cluster import KMeans
import statsmodels.api as sm
from scipy.stats import f
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from sklearn.cluster import DBSCAN

from scipy.stats import expon
from scipy.optimize import curve_fit
import scipy.stats as ss
from scipy.optimize import minimize

from scipy.linalg import inv, pinv, eigh

from scipy.signal import argrelextrema
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d

import pickle

from posixpath import split
# build connections by lane
import networkx as nx
from scipy.spatial import cKDTree

import numpy as np
from scipy.ndimage import uniform_filter1d
import networkx as nx
from shapely.geometry import LineString, box

# Read Data

In [ ]:
# select one of these ["US101", "I80", "I395", "I90_94_2", "I90_94_4" , "I90_94_6"] # US101, I80, I395, and three runs of I90 datasets
# need to change the directory before you store the data
traj_data_dir = "processed_data/"
site_name = "I395"


if site_name == "US101":
  #NGSIM US101
  rel_data=pd.read_csv(traj_data_dir + "US101_processed.csv")
  suffix =  "US101"
  
  max_t=2600 # 700
  min_t=0
  min_x=50
  max_x=650

  lanes=[1,2,3,4,5]

if site_name == "I80":
  
  rel_data=pd.read_csv(traj_data_dir +"I80_processed.csv")
  suffix="I80"

  # I80
  max_t=1500 # 700
  min_t=0
  min_x=50
  max_x=500
    
  lanes=[1,2,3,4,5,6]

if site_name == "I395":
  rel_data=pd.read_csv(traj_data_dir + "I395_processed.csv")
  suffix="I395"

  max_t=7000
  min_t=0
  min_x=100
  max_x=450
  
  lanes=sorted(rel_data["lane-kf"].unique())

if site_name in ["I90_94_2", "I90_94_4", "I90_94_6"]:
  run = site_name.split("_")[-1]
  run_name = run
  suffix = "I90_94_"+str(run)
  rel_data = pd.read_csv(traj_data_dir + "I90_94_run"+str(run)+"_processed.csv")

  # this is for TGSIM
  max_t=700
  min_t=0
  min_x=100
  max_x=450
  
  lanes=sorted(rel_data["lane-kf"].unique())


# find the slowdowns

## Definition of Required Functions

In [ ]:
# necessary function to check if the connections cross with each other
def ccw(tA, xA, tB, xB, tC, xC):
    return (xC - xA) * (tB - tA) > (xB - xA) * (tC - tA)


def check_crossing(connection1, other_connections):
    t1, x1 = connection1[0]
    t2, x2 = connection1[1]

    t3 = other_connections[0]
    x3 = other_connections[1]
    t4 = other_connections[2]
    x4 = other_connections[3]

    endpoint_match = (
        ((t1 == t3) & (x1 == x3)) |
        ((t1 == t4) & (x1 == x4)) |
        ((t2 == t3) & (x2 == x3)) |
        ((t2 == t4) & (x2 == x4))
    )

    cond1 = ccw(t1, x1, t3, x3, t4, x4) != ccw(t2, x2, t3, x3, t4, x4)
    cond2 = ccw(t1, x1, t2, x2, t3, x3) != ccw(t1, x1, t2, x2, t4, x4)
    #print(cond1 & cond2 & ~endpoint_match)
    return np.any(cond1 & cond2 & ~endpoint_match)



In [ ]:
# Determine if the collected wave segment is out of bound. If so, clip it onto the boundary
def is_out_of_bound(t,x):
  if x<min_x+margin_x or x>max_x-margin_x or t<min_t+margin_t or t>max_t-margin_t:
    return True
  else:
    return False



In [ ]:
hyperparams = [[80, 5, 50, 4] ] # The first one is baseline: percentile speed compared tod ataset, dt, dx , number of forward or backward 

case = 0
hyperparam = hyperparams[case]

percentage_spd_drop = hyperparam[0]
connect_t = hyperparam[1]
connect_x = hyperparam[2]
num_neis = hyperparam[3]


In [ ]:

# Hyper parameters defined

# If the detected shockwave segment falls out of this margin,
# we consider it to be censored, because the boundary may skew our extraction process:
margin_t=25
margin_x=25
# the rectangle of the bonding boxes
rect = box(min_t+margin_t-0.1, min_x+margin_x-0.1, max_t-margin_x+0.1, max_x-margin_x+0.1)

# define the speed threshold. If it is above this, it would not be considered shockwave in our process even if there is deceleration
all_spds=rel_data["speed-kf"].values
spd_thresh = np.percentile(all_spds, percentage_spd_drop )
spd_thresh


max_connect_t=connect_t # the follower' slowdown point cannot occur more than this value later, otherwise they are not connected
max_connect_x=connect_x # the follower' slowdown point cannot occur more than this value away spatially, otherwise they are not connected

# tolerances
num_look_backward = num_neis # if the first follower does not experience this slowdown, how many further ones do we allow to look into?
num_look_forward = num_neis # if the first leader does not experience this slowdown, how many further ones do we allow to look into?


min_wave_spd = -20 # the minimum slope of the connection between leader-follow pair.
max_wave_spd = spd_thresh # the maximum slope of the connection between leader-follower pair: definitely no more than the vehicle speed. But in practice this is usually not activated
sm_wd = 20 # the window size used to smoothen the acceleration.

#suffix = "I90_94_"+str(run)+"_case_"+str(case)

In [ ]:
# run this to form connections (Unsmoothed)

all_connections_by_lane={} # each element is a tuple representing time range and location range.
# these save the speed, acceleration, and ID of the slowdown points
get_local_min_vs_by_lane={}
get_local_min_ids_by_lane={}
get_local_min_accs_by_lane={}

# if there is no follower or leader whose slowdown points satisfy the requirements, keep looking until one is found
for lane in lanes:
    all_connections_by_lane[lane]=[]
    get_local_min_vs_by_lane[lane]={}
    get_local_min_ids_by_lane[lane]={}
    get_local_min_accs_by_lane[lane]={}
    print("Identify Slow", lane)
    all_local_min_xs=[]
    all_local_min_ts=[]
    all_local_min_vs=[]
    all_local_min_accs=[]
    all_local_min_veh_ids=[]
    all_local_min_leader_ids=[]
    all_local_min_follower_ids=[]

    lane_data=rel_data[( rel_data["lane-kf"]==lane)  ]
    #lane_data["time"]=np.round(lane_data["time"], 1)
    #lane_data["dist_cntr"]=np.round(lane_data["dist_cntr"], 1)

    plt.figure(figsize=((max_t-min_t)/20,6))
    color_bar = np.maximum(np.minimum(lane_data["speed-kf"].values, 20*np.ones(len(lane_data))), 0*np.ones(len(lane_data)))
    plt.scatter(lane_data["time"], lane_data["dist_cntr"], s=0.1, c=color_bar, cmap='RdYlGn')

    ids=lane_data["ID"].unique()

    all_next_nodes={} # the connections when finding the leader, can be multiple
    all_last_node={} # can just be one
    all_connections = []
    all_leader_ids=[]
    all_follower_ids=[]
    candidate_follower_ids=[]
    all_veh_ids=[] # regardless of leader or follower



    # first, obtain the accelerations in the dataset
    for id in ids:
      veh=lane_data[lane_data["ID"]==id]
      if len(veh)<20:
        continue

      times = veh["time"].values
      xs = veh["dist_cntr"].values
      spds = veh["speed-kf"].values # adjust here for other datasets, vx for ngsim speed-kf for tgsi

      accs=np.gradient(spds)/0.1

      leaders= veh["leader_id"].values
      followers = veh["follower_id"].values

      t_diff = times[1:] - times[:-1]
      split_idxs = np.where(t_diff>0.11)[0]+1
      #accs_split = np.split(accs, split_idxs)
      times_split = np.split(times, split_idxs)
      xs_split = np.split(xs, split_idxs)
      spds_split = np.split(spds, split_idxs)
      leaders_split = np.split(leaders, split_idxs)
      followers_split = np.split(followers, split_idxs)

      # split based on time discontinity: sometimes vehicles may lane change and we need to segment them
      for i in range(len(xs_split)):
        if len(xs_split[i])<50:
          continue

        new_accs= np.gradient(spds_split[i])/0.1
        #plt.plot(xs_split[i], new_accs)
        sm_accs = gaussian_filter1d(new_accs, sigma=sm_wd) #savgol_filter(new_accs, window_length=19, polyorder=1)


        # find local min acceleration that is less than 0 with relatively low speeds, forming the slowdown points.
        local_min_indices = argrelextrema(sm_accs, np.less)[0]
        negative_acc_indices = np.where(sm_accs<0)[0]
        low_spd_indices = np.where(spds_split[i]<spd_thresh)[0]
        if len(local_min_indices)==0:
          local_min_indices = np.where(sm_accs == min(sm_accs))[0]
        low_acc_indices = np.intersect1d(local_min_indices, negative_acc_indices)
        low_acc_indices = np.intersect1d(low_acc_indices, low_spd_indices)
        negative_acc_indices = np.intersect1d(negative_acc_indices, low_spd_indices)


        slowdown_xs=xs_split[i][low_acc_indices]
        slowdown_ts=times_split[i][low_acc_indices]
        slowdown_accs = sm_accs[low_acc_indices]
        slowdown_spds = spds_split[i][low_acc_indices]
        slowdown_leaders=leaders_split[i][low_acc_indices]
        slowdown_followers=followers_split[i][low_acc_indices]


        all_local_min_ts.append(slowdown_ts)
        all_local_min_xs.append(slowdown_xs)
        all_local_min_vs.append(slowdown_spds)
        all_local_min_accs.append(slowdown_accs)
        all_local_min_veh_ids.append([id]*len(slowdown_xs))
        all_local_min_leader_ids.append(slowdown_leaders)
        all_local_min_follower_ids.append(slowdown_followers)
        all_dec_xs=xs_split[i][negative_acc_indices]
        all_dec_ts=times_split[i][negative_acc_indices]
        all_dec_accs =sm_accs[negative_acc_indices]

        plt.scatter(slowdown_ts, slowdown_xs, color="grey", s=5, zorder=2)

    all_local_min_ts = np.concatenate(all_local_min_ts)
    all_local_min_xs = np.concatenate(all_local_min_xs)
    all_local_min_vs = np.concatenate(all_local_min_vs)
    all_local_min_accs = np.concatenate(all_local_min_accs)
    all_local_min_veh_ids = np.concatenate(all_local_min_veh_ids)
    all_local_min_leader_ids = np.concatenate(all_local_min_leader_ids)
    all_local_min_follower_ids = np.concatenate(all_local_min_follower_ids)
    print("num", len(all_local_min_follower_ids))

    for i in range(len(all_local_min_ts)):

      t=all_local_min_ts[i]
      x=all_local_min_xs[i]
      v=all_local_min_vs[i]
      a=all_local_min_accs[i]
      get_local_min_vs_by_lane[lane][(t,x)]=v
      get_local_min_ids_by_lane[lane][(t,x)]=id
      get_local_min_accs_by_lane[lane][(t,x)]=a
      if a>0:
        print(a)

      id=all_local_min_veh_ids[i]
      data_t = lane_data[np.absolute(lane_data["time"]-t)<=0.01]
      sorted_idx = np.argsort(data_t["dist_cntr"])
      sorted_ids=data_t["ID"].values[sorted_idx]
      id_idx = np.where(sorted_ids==id)[0][0]
      leader_ids = sorted_ids[id_idx+1:] # higher idx more into front
      follower_ids = sorted_ids[:id_idx] # lower idx more at back


      # from follower to leader
      if len(leader_ids)>=1:
        iter = 0
        while True:
          leader_id = leader_ids[iter]
          connect_spds = (all_local_min_xs-x)/(all_local_min_ts-t+0.01)
          to_connect_idxes_leader = np.where((all_local_min_ts-t<0) & (t- all_local_min_ts<max_connect_t*(iter+1)) &  ( np.absolute(all_local_min_xs-x)<=max_connect_x ) & (all_local_min_veh_ids== leader_id) & (connect_spds<max_wave_spd) & (connect_spds>min_wave_spd) )[0]
          if len(to_connect_idxes_leader)>0:
            min_connect_t = min(np.absolute(t-all_local_min_ts[to_connect_idxes_leader]))
            #to_connect_idxes_leader = np.where((t-all_local_min_ts==min_connect_t) & (all_local_min_veh_ids== leader_id) & ((x-all_local_min_xs)/(t-all_local_min_ts+0.01)>-20) )[0]
            to_connect_idxes_leader = np.where((all_local_min_ts-t<0) & (np.absolute(t-all_local_min_ts)==min_connect_t) &  ( np.absolute(all_local_min_xs-x)<=max_connect_x ) & (all_local_min_veh_ids== leader_id) & (connect_spds<max_wave_spd) & (connect_spds>min_wave_spd) )[0]
            if len(to_connect_idxes_leader)>0:
              leader_x = all_local_min_xs[to_connect_idxes_leader[0]]
              leader_t = all_local_min_ts[to_connect_idxes_leader[0]]
              leader_v = all_local_min_vs[to_connect_idxes_leader[0]]
              leader_acc = all_local_min_accs[to_connect_idxes_leader[0]]


              connections_reshaped = np.transpose(np.array(all_connections).reshape(-1,4))

              crossing_exist = check_crossing([(t,x), (leader_t, leader_x)], connections_reshaped)
              #print(crossing_exist)
              if not crossing_exist:
                if leader_t > t:
                  print("Issue in leader connection")
                all_connections.append([(leader_t,leader_x), (t,x)])
                all_leader_ids.append(leader_id)
                all_follower_ids.append(id) # itsets
                candidate_follower_ids.append(follower_ids)       # the follower of iteself
                if not (leader_t, leader_x) in all_next_nodes:
                  all_next_nodes[(leader_t, leader_x)]=[]
                all_next_nodes[(leader_t, leader_x)].append((t,x))

                all_last_node[(t,x)]=(leader_t, leader_x)

                break

          iter=iter+1
          if iter>=len(leader_ids) or iter>=num_look_forward:
            break

    # now clean connections

    cleaned_connections=[]
    cleaned_leader_ids=[]
    cleaned_follower_ids=[]
    cleaned_ff_ids=[]
    cleaned_last_nodes={}
    cleaned_next_nodes={}
    cleaned_follower_nodes=[]
    cleaned_nodes=[]
    for i, connection in enumerate(all_connections):
      leader_id = all_leader_ids[i]
      follower_id = all_follower_ids[i]
      ff_ids = candidate_follower_ids[i]
      leader_t,leader_x = connection[0]
      follower_t,follower_x = connection[1]
      # if the leader_node is a junction and follower node terminates, do not plot
      valid_flag=True
      if (len(all_next_nodes[(leader_t, leader_x)])> 1 or not (leader_t, leader_x) in all_last_node ) and (not (follower_t, follower_x) in all_next_nodes ):
        #plt.plot([leader_t, follower_t], [leader_x, follower_x], color="grey", zorder=1)
        valid_flag=False
        continue
      # if the leader is a start node and the follower is a junction or am end node, do not plot (make sure the leader node is only connected to a follower)
      elif (not (leader_t, leader_x) in all_last_node and len(all_next_nodes[(leader_t, leader_x)])==1 ) and (not (follower_t, follower_x) in all_next_nodes or len(all_next_nodes[(follower_t, follower_x)])>1 ):
        valid_flag=False
        #plt.plot([leader_t, follower_t], [leader_x, follower_x], color="grey", zorder=1)
        continue
      # also do not plot those that have a size of 2
      # the last node is a start the next next node is a junctiom or end
      #'''
      elif (not (leader_t, leader_x) in all_last_node and len(all_next_nodes[(leader_t, leader_x)])==1 ) and ((follower_t, follower_x) in all_next_nodes):
        next_next_node = all_next_nodes[(follower_t, follower_x)][0]
        if ((next_next_node in all_next_nodes) and len(all_next_nodes[next_next_node])>1) or (not next_next_node in all_next_nodes) :
          valid_flag=False
          #plt.plot([leader_t, follower_t], [leader_x, follower_x], color="blue", zorder=1)
          continue

      # the last last node is a start and the next node is a junction
      elif (leader_t, leader_x) in all_last_node and (follower_t, follower_x) in all_next_nodes and len(all_next_nodes[(follower_t, follower_x)])>1:
        last_last_node = all_last_node[(leader_t, leader_x)]
        if not (last_last_node in all_last_node) and len(all_next_nodes[last_last_node])==1  :
          valid_flag=False
          #plt.plot([leader_t, follower_t], [leader_x, follower_x], color="blue", zorder=1)
          continue
      # the last node is the junction and next next is end
      elif (leader_t, leader_x) in all_next_nodes and len(all_next_nodes[(leader_t, leader_x)])>1 and (follower_t, follower_x) in all_next_nodes and len(all_next_nodes[(follower_t, follower_x)])==1:
        next_next_node = all_next_nodes[(follower_t, follower_x)][0]
        if not next_next_node in all_next_nodes:
          valid_flag=False
          #plt.plot([leader_t, follower_t], [leader_x, follower_x], color="blue", zorder=1)
          continue

      # the last last node is junction or end and next node is the end
      elif (leader_t, leader_x) in all_last_node and len(all_next_nodes[(leader_t, leader_x)])==1 and (not (follower_t, follower_x) in all_next_nodes):
        last_last_node = all_last_node[(leader_t, leader_x)]
        if (not last_last_node in all_last_node and len(all_next_nodes[last_last_node])==1  ) or ( len(all_next_nodes[last_last_node])>1) :
          valid_flag=False
          #plt.plot([leader_t, follower_t], [leader_x, follower_x], color="blue", zorder=1)
          continue
      #'''

      if valid_flag:
        if leader_t > follower_t:
          print("time Issue when Filtering leader")
        cleaned_connections.append(connection)
        cleaned_follower_ids.append(follower_id)
        cleaned_leader_ids.append(leader_id)
        cleaned_ff_ids.append(ff_ids)
        cleaned_follower_nodes.append((follower_t, follower_x))
        if not (leader_t, leader_x) in cleaned_next_nodes:
          cleaned_next_nodes[(leader_t, leader_x)]=[]
        cleaned_next_nodes[(leader_t, leader_x)].append((follower_t, follower_x))
        if not (follower_t, follower_x) in cleaned_last_nodes:
          cleaned_last_nodes[(follower_t, follower_x)]=[]
        cleaned_last_nodes[(follower_t, follower_x)].append((leader_t, leader_x))
        if not (leader_t, leader_x) in cleaned_nodes:
          cleaned_nodes.append((leader_t, leader_x))
          all_veh_ids.append(leader_id)
        if not (follower_t, follower_x) in cleaned_nodes:
          cleaned_nodes.append((follower_t, follower_x))
          all_veh_ids.append(follower_id)


    # for points with no follower, we find the follower, so that we capture merging
    # find connection to the follower for ending nodes
    all_wave_ts = np.transpose(cleaned_nodes)[0]
    all_wave_xs = np.transpose(cleaned_nodes)[1]
    all_follower_ts= np.transpose(cleaned_follower_nodes)[0]
    all_follower_xs= np.transpose(cleaned_follower_nodes)[1]
    all_veh_ids = np.array(all_veh_ids)
    for i, veh_id in enumerate(cleaned_follower_ids): # only
      node = cleaned_follower_nodes[i]
      if node in cleaned_next_nodes:
        continue

      t = node[0]
      x = node[1]
      ff_ids = cleaned_ff_ids[i]
      # wave xs and ts comparison

      if len(ff_ids)>=1:
        iter = 0
        while True:
          ff_id = ff_ids[-(iter+1)]
          connect_spds = (x-all_wave_xs)/(t-all_wave_ts+0.01)
          to_connect_idxes_follower = np.where((all_wave_ts-t>0) & (all_wave_ts-t<max_connect_t*(iter+1)) & ( np.absolute(all_wave_xs-x)<=max_connect_x ) & (all_veh_ids == ff_id) & (connect_spds<max_wave_spd) & (connect_spds>min_wave_spd)  )[0]
          if len(to_connect_idxes_follower)>0:

            min_connect_t = min(np.absolute(all_wave_ts[to_connect_idxes_follower]-t))
            to_connect_idxes_follower = np.where((all_wave_ts-t>0) & (np.absolute(all_wave_ts-t)==min_connect_t) &  ( np.absolute(all_wave_xs-x)<=max_connect_x ) & (all_veh_ids== ff_id) & (connect_spds<max_wave_spd) & (connect_spds>min_wave_spd))[0]

            if len(to_connect_idxes_follower)>0:
              #plt.plot([all_local_min_ts[i], all_local_min_ts[to_connect_idxes_follower[j]]], [all_local_min_xs[i], all_local_min_xs[to_connect_idxes_follower[j]]], color="black", zorder=1)
              ff_x = all_wave_xs[to_connect_idxes_follower[0]]
              ff_t = all_wave_ts[to_connect_idxes_follower[0]]

              #plt.plot([t, ff_t], [x, ff_x], color="blue", zorder=1)
              # make sure there is no crossing

              connections_reshaped = np.transpose(np.array(cleaned_connections).reshape(-1,4))
              crossing_exist = check_crossing([(t,x), (ff_t, ff_x)], connections_reshaped)
              if not crossing_exist:
                if t > ff_t:
                  print("Issue in following connection")

                cleaned_connections.append([(t,x), (ff_t, ff_x)])
                if not (t, x) in cleaned_next_nodes:
                  cleaned_next_nodes[(t, x)]=[]
                cleaned_next_nodes[(t, x)].append((ff_t,ff_x))
                if not (ff_t, ff_x) in cleaned_last_nodes:
                  cleaned_last_nodes[(ff_t, ff_x)]=[]
                cleaned_last_nodes[(ff_t, ff_x)].append((t, x))

                break

          iter = iter+1
          if iter>=len(ff_ids) or iter>=num_look_backward:
            break

    # reconstruct the graph and filter
    # get the degree of the each of the nodes
    node_degree_dict={}
    node_is_junction={}
    for node in cleaned_next_nodes:
      if not node in cleaned_last_nodes:
        node_degree_dict[node]=len(cleaned_next_nodes[node])
        if len(cleaned_next_nodes[node])>1:
          node_is_junction[node]=1
        else:
          node_is_junction[node]=0
      else:
        node_degree_dict[node]=len(cleaned_next_nodes[node])+len(cleaned_last_nodes[node])
        if len(cleaned_next_nodes[node])>1 or len(cleaned_last_nodes[node])>1:
          node_is_junction[node]=1
        else:
          node_is_junction[node]=0
    for node in cleaned_last_nodes:
      if node in node_degree_dict:
        continue
      if node in cleaned_next_nodes:
        node_degree_dict[node]=len(cleaned_last_nodes[node])+len(cleaned_next_nodes[node])
        if len(cleaned_next_nodes[node])>1 or len(cleaned_last_nodes[node])>1:
          node_is_junction[node]=1
        else:
          node_is_junction[node]=0
      else:
        node_degree_dict[node]=len(cleaned_last_nodes[node])
        if len(cleaned_last_nodes[node])>1:
          node_is_junction[node]=1
        else:
          node_is_junction[node]=0

    for connection in cleaned_connections:
      node0 = connection[0]
      node1 = connection[1]

      valid_connection = True
      # the same logic, no branched on length 1
      #'''
      if node_degree_dict[node0]==1 and node_is_junction[node1]==1:
        valid_connection = False
      elif node_degree_dict[node1]==1 and node_is_junction[node0]==1:
        valid_connection = False
      elif node_degree_dict[node1]==1 and node_degree_dict[node0]==1 :
        valid_connection = False
      #'''
      #if node_degree_dict[node0]!=2 and node_degree_dict[node1]!=2:
        #valid_connection = False


      # now make sure no braches of length 2
      elif (not node0 in cleaned_last_nodes) and node_degree_dict[node1]==2 and node_is_junction[node1]==0:
        next_next_node = cleaned_next_nodes[node1][0]
        if node_degree_dict[next_next_node]==1 or node_is_junction[next_next_node]==1: # also consider seperate branches of length 2
          valid_connection = False
      elif node_degree_dict[node0]==2 and node_is_junction[node0]==0 and node_is_junction[node1]==1:
        last_last_node = cleaned_last_nodes[node0][0]
        if node_degree_dict[last_last_node]==1 or (not last_last_node in cleaned_last_nodes): # or no last node
          valid_connection = False
      elif node_is_junction[node0]==1 and node_is_junction[node1]==0 and node_degree_dict[node1]==2:
        next_next_node = cleaned_next_nodes[node1][0]
        if node_degree_dict[next_next_node]==1 or (not next_next_node in cleaned_next_nodes): # or no next node
          valid_connection = False
      elif node_is_junction[node0]==0 and node_degree_dict[node0]==2 and (not node1 in cleaned_next_nodes):
        last_last_node = cleaned_last_nodes[node0][0]
        if node_degree_dict[last_last_node]==1 or node_is_junction[last_last_node]==1: # also consider seperate branches of length 2
          valid_connection = False

      if valid_connection:


        # I have to truncate the part out of the boundary
        node0_t=node0[0]
        node0_x=node0[1]
        node1_t=node1[0]
        node1_x=node1[1]

        if node0_t>node1_t:
          print("Issue when filtering follower")


        # if both are out of bound
        line = LineString([(node0_t, node0_x), (node1_t, node1_x)])
        if is_out_of_bound(node0_t, node0_x) and is_out_of_bound(node1_t, node1_x):
          continue
        elif is_out_of_bound(node0_t, node0_x) and not is_out_of_bound(node1_t, node1_x):
          # find the intersection
          intersection = line.intersection(rect)
          coord=list(intersection.coords)
          node0_t, node0_x = coord[0]
          #print("0", node0_t, node0_x)
          #print(coord)
          #print()
        elif not is_out_of_bound(node0_t, node0_x) and is_out_of_bound(node1_t, node1_x):
          intersection = line.intersection(rect)
          coord=list(intersection.coords)
          node1_t, node1_x = coord[1]


        if node0_t> node1_t:
          print("Issue when plotting")

        plt.plot([node0_t, node1_t], [node0_x, node1_x], color="black", zorder=1)
        plt.scatter(node0_t, node0_x, color="red", s=10, zorder=2)
        plt.scatter(node1_t, node1_x, color="red", s=10, zorder=2)
        new_connection = [(np.round(node0_t,1), node0_x), (np.round(node1_t,1), node1_x)]
        #cleaned_connections.append(new_connection)

        all_connections_by_lane[lane].append(new_connection) # this is the only thing uses

    plt.plot([min_t+margin_t, max_t-margin_t,  max_t-margin_t, min_t+margin_t, min_t+margin_t  ], [min_x+margin_x, min_x+margin_x, max_x-margin_x, max_x-margin_x,  min_x+margin_x])
    plt.xlabel("time (s)")
    plt.ylabel("Location (m)")
    plt.show()

In [ ]:
wave_dir = "extracted_wave_data/"

In [ ]:
suffix

In [ ]:
# save the data collected
#'''
np.save(wave_dir + "all_connections_by_lane_"+suffix+".npy", all_connections_by_lane, allow_pickle=True)
np.save(wave_dir + "get_local_min_vs_by_lane_"+suffix+".npy", get_local_min_vs_by_lane, allow_pickle=True)
np.save(wave_dir + "get_local_min_ids_by_lane_"+suffix+".npy", get_local_min_ids_by_lane, allow_pickle=True)
np.save(wave_dir + "get_local_min_accs_by_lane_"+suffix+".npy", get_local_min_accs_by_lane, allow_pickle=True)
#'''

# Others_to_save

In [ ]:
# get the degree of nodes, which are represented by (time, location)
nodes_num_in_by_lane={}
nodes_num_out_by_lane={}
next_nodes_by_lane={}
last_nodes_by_lane={}
for lane in lanes:
  node_num_out={}
  node_num_in={}
  next_nodes={}
  last_nodes={}
  all_connections = all_connections_by_lane[lane]
  for connection in all_connections:
    node0 = connection[0]
    node1 = connection[1]
    if not node0 in node_num_out:
      node_num_out[node0]=0
      next_nodes[node0]=[]
    if not node0 in node_num_in:
      node_num_in[node0]=0
      last_nodes[node0]=[]
    if not node1 in node_num_out:
      node_num_out[node1]=0
      next_nodes[node1]=[]
    if not node1 in node_num_in:
      node_num_in[node1]=0
      last_nodes[node1]=[]

    node_num_out[node0]=node_num_out[node0]+1
    node_num_in[node1]=node_num_in[node1]+1
    next_nodes[node0].append(node1)
    last_nodes[node1].append(node0)

  next_nodes_by_lane[lane]=next_nodes
  last_nodes_by_lane[lane]=last_nodes


  nodes_num_in_by_lane[lane]=node_num_in
  nodes_num_out_by_lane[lane]=node_num_out


In [ ]:
#'''
np.save(wave_dir + "nodes_num_in_by_lane_"+suffix+".npy", nodes_num_in_by_lane, allow_pickle=True)
np.save(wave_dir + "nodes_num_out_by_lane_"+suffix+".npy", nodes_num_out_by_lane, allow_pickle=True)
np.save(wave_dir + "next_nodes_by_lane_"+suffix+".npy", next_nodes_by_lane, allow_pickle=True)
np.save(wave_dir + "last_nodes_by_lane_"+suffix+".npy", last_nodes_by_lane, allow_pickle=True)
#'''

In [ ]:
# the function to find the next node of each of the nodes
def find_next_event_nodes(start_node, next_nodes, last_nodes):
  ini_next_nodes=next_nodes[start_node]
  all_paths_next={}
  for ini_next_node in ini_next_nodes:
    nodes_traveled=[start_node]
    next_node = ini_next_node
    while len(next_nodes[next_node])==1 and len(last_nodes[next_node])==1:
      nodes_traveled.append(next_node)
      next_node = next_nodes[next_node][0]
    end_node = next_node
    nodes_traveled.append(end_node)
    if not end_node in all_paths_next :
      all_paths_next[end_node]=[]
    all_paths_next[end_node].append(nodes_traveled)

  return all_paths_next

In [ ]:
# function to smoothen the portion between the event nodes, fixing the time and location of the start and end nodes
def interpolate_paths(path_ts, path_xs, dt=0.1, wd=20):

  new_ts = np.arange(path_ts[0], path_ts[-1]+dt, dt)

  # Perform linear interpolation
  new_xs = np.interp(new_ts, path_ts, path_xs)
  #print(new_xs[-1], path_xs[-1])

  # smoothen them with variable moving wd
  sm_xs=[]
  N = len(new_xs)
  if N>wd:
    thresh1=wd // 2
    thresh2=N-wd // 2
  else:
    thresh1=N//2
    thresh2=N//2
  for i in range(len(new_xs)):
    if i < thresh1:
        # Expand symmetrically from center i as much as possible
        start = 0
        end = min(N, 2 * i + 1)
    elif i >=  thresh2:
        # Mirror the shrinking window at the end
        start = max(0, 2*i- (N)+1)
        end = N
    else:
        start = i - wd // 2
        end = i + wd // 2 + 1
    sm_xs.append(np.mean(new_xs[start:end]))

  return new_ts, sm_xs

In [ ]:
# between each pair of event nodes, we find the slowdown points the trajectory of the shockwave segment has gone through
paths_between_event_nodes_by_lane={} # lane, start_node, end_node, list of paths

for lane in lanes:

  nodes_num_out=nodes_num_out_by_lane[lane]
  nodes_num_in=nodes_num_in_by_lane[lane]
  paths_between_event_nodes_by_lane[lane]={}

  count =0

  for node in nodes_num_out:
    if (nodes_num_in[node]!=1 and nodes_num_out[node]>0 ) or nodes_num_out[node]>1:
      count = count + 1
      all_paths_next=find_next_event_nodes(node, next_nodes_by_lane[lane], last_nodes_by_lane[lane])
      paths_between_event_nodes_by_lane[lane][node]=all_paths_next
      # start node, merge node or diverge node:
      # start tracing
  print(count)

np.save(wave_dir + "paths_between_event_nodes_by_lane_"+suffix+".npy", paths_between_event_nodes_by_lane, allow_pickle=True)

In [ ]:
connections_between_event_nodes_by_lane={}
for lane in lanes:
  lane_data=rel_data[( rel_data["lane-kf"]==lane)  ]
    #lane_data["time"]=np.round(lane_data["time"], 1)
    #lane_data["dist_cntr"]=np.round(lane_data["dist_cntr"], 1)

  plt.figure(figsize=((max_t-min_t)/20,6))
  color_bar = np.clip(lane_data["speed-kf"].values, 0, 20)
  sc = plt.scatter(lane_data["time"], lane_data["dist_cntr"], s=1, c=color_bar, vmin =0, vmax =20, cmap='RdYlGn')
  cbar = plt.colorbar(sc, ticks=[0, 5, 10, 15, 20])
  cbar.ax.tick_params(labelsize=15)
  cbar.set_label("Speed (m/s)", fontsize=15)
  #lane_data=data_by_lane[lane]


  connections_between_event_nodes_by_lane[lane]={}

  paths_between_event_nodes = paths_between_event_nodes_by_lane[lane]
  for start_node in paths_between_event_nodes:
    connections_between_event_nodes_by_lane[lane][start_node]={}
    for end_node in paths_between_event_nodes[start_node]:
      connections_between_event_nodes_by_lane[lane][start_node][end_node]=[]
      for path in paths_between_event_nodes[start_node][end_node]:
        path_xs=np.transpose(path)[1]
        path_ts=np.transpose(path)[0]
        new_ts, new_xs = interpolate_paths(path_ts, path_xs, wd=50)
        new_xs=np.round(new_xs,1)
        #if new_xs[-1]!=path_xs[-1]:

          #print(new_xs[-1], path_xs[-1])
        plt.plot(new_ts, new_xs, color="grey", zorder=2, linewidth = 3)
        plt.scatter(new_ts[0], new_xs[0], color="black", s=20, zorder=2)
        plt.scatter(new_ts[-1], new_xs[-1], color="black", s=20, zorder=2)

        #plt.plot(path_ts, path_xs, color="red", zorder=1)
        connections_between_event_nodes_by_lane[lane][start_node][end_node].append((new_ts, new_xs))

        #
  plt.title("Lane"+str(lane), fontsize=20)
  #plt.xlim(0,700)
  #plt.ylim(100, 450)
  plt.xticks(fontsize=15)
  plt.yticks(fontsize=15)
  plt.xlabel("Time (s)", fontsize=15)
  plt.ylabel("Location (m)", fontsize=15)

  plt.show()

In [ ]:
# save the smoothed connections
np.save(wave_dir + "connections_between_event_nodes_by_lane_"+suffix+".npy", connections_between_event_nodes_by_lane, allow_pickle=True)